# Los programadores inteligentes escriben código tonto

-----------------------

Un error común entre novatos es escribir código complejo. Al usar alguna funcionalidad avanzada o novedosa del lenguaje o simplemente para demostrar la capacidad del autor (*mira que seré listo, que eres incapaz de entender lo que he escrito*) el autor comete uno de los peores pecados: su código deja de ser **claro y sencillo**.

El **código confuso es peor que el código incorrecto**, porque lo incorrecto y claro se puede arreglar fácilmente, mientras que lo confuso sólo se puede tirar a la basura.

Ese código *listo* no demuestra que seas un gran programador, sino todo lo contrario. Recuerda la frase lapidaria de [Terry Davies](https://thenewstack.io/the-troubled-legacy-of-terry-davis-gods-lonely-programmer/), creador de un sistema operativo para comunicarse con Dios (sic):

> *“An idiot admires complexity, a genius admires simplicity (...)”*
> 
> El *programador de Dios* que vivía en un infierno, tenía mucha razón.

Sé un genio y recuerda que **los programadores listos escriben código tonto**. 

Código *obvio, claro, sin recovecos*, que cualquier idiota sería capaz de entender y modificar. Esa es tu labor.

## Condiciones booleanas complejas

Un caso muy común es usar condiciones booleanas demasiado complejas. No lo hagas.

Veamos un ejemplo concreto. Supongamos que estás trabajando en un sistema de reservas de hoteles y tienes que decidir si se le envía un email de notificación a un usuario o no. Te encuentras con el siguiente código que te ha dejado el anterior programador antes de ser despedido (¿por qué habrá sido?):




```Python
if ((reservationId and notification.reservationId == reservationId) or
    (facilityId and notification.facilityId in facilityIds) or
    (hotelId and notification.hotelId in hotelIds) and 
    (hotelUser and (notification.type.toAllHotelUsers or 
                    (reservationId and notification.type.toAllReservations))) or
    (isAdmin and notification.hotelId in hotelIds)) and 
    (userId != notification.authorId or notification.authorId is None):
    
    send(notification)
```



![](cthulhu.jpg)

Eso es un horror absolutamente inmanejable, sacado de las profundidades del código fuente del Cthlhhu. Si las pesadillas de [H.R. Giger](https://www.hrgigermuseum.com/) estuviesen escritas en Python, ese `if` sería parte de ellas.

### ¿Cómo lo resolvemos?

Con los Pilares de la Ciberkinesis. En concreto con *DIVIDE Y VENCERÁS*.  

1. Rompe esa condición monstruosa en varias más sencillas.
2. A cada una dale un nombre adecuado
3. No hagas conversiones implícitas de booleanos, compara explicitamente con None.
4. Mantén la condición del `if` sencilla.


Haremos las siguientes modificaciones:

1. **Agrupación lógica de condiciones**: Al dividir el código en secciones lógicas y agrupar condiciones relacionadas, el flujo de lectura mejora.
2. **Nombres de variables más descriptivos**: Asegúrate de que los nombres de las variables sean claros y representen exactamente lo que verifican.
3. **Eliminación de condiciones redundantes**: Si algunas condiciones se pueden unificar o si una condición hace innecesaria otra, se deben eliminar las redundancias.
4. **Eliminar conversiones implícitas a bool**: En vez `if userId:` haremos `if userId is not None:`

#### Código mejorado

```Python
# Verifica si el remitente no es el receptor o el autor es desconocido
sender_is_not_receiver = userId != notification.authorId or notification.authorId is None

# Verifica si la notificación coincide con una reserva, instalación u hotel
reservation_matches = reservationId is not None and notification.reservationId == reservationId
facility_matches = facilityId is not None and notification.facilityId in facilityIds
hotel_matches = hotelId is not None and notification.hotelId in hotelIds

# Verifica si la notificación está dirigida a todos los usuarios o si el usuario es admin
addressed_to_all = hotelUser is not None and (notification.type.toAllHotelUsers or 
                                              (reservationId is not None and notification.type.toAllReservations))
admin_access = isAdmin and notification.hotelId in hotelIds

# Combina todas las condiciones para verificar si se debe enviar la notificación
should_send = sender_is_not_receiver and (reservation_matches or facility_matches or (hotel_matches and (addressed_to_all or admin_access)))

if should_send:
    send(notification)
```




#### Resultado

- **Claridad**: Agrupar condiciones relacionadas en variables descriptivas ayuda a entender qué verifica cada parte del código.
- **Modularidad**: Dividir el código en pequeñas secciones lógicas hace que cada paso del proceso sea más fácil de seguir.
- **Legibilidad**: El uso de nombres de variables claros y la eliminación de redundancias hacen que el código sea más fácil de mantener y menos propenso a errores.

Sigue siendo complejo, porque las condiciones que determinan si se debe de enviar una notificación o no, lo son. No podemos modificar eso, puesto que son las reglas del hotel o del sistema de reservas, pero al menos la complejidad está rota en pequeños trozos más manejables y comprehensibles.



### Conclusión

**ESCRIBE CÓDIGO TONTO**

### `!= None` vs `is not None`

Uno de los cambios que hemos hecho ha sido comparar de forma explícita alguna de las variables con `None`. Sin embargo, no lo hicimos de la forma habitual hasta ahora, con el *operador de desigualdad* `!=`.

En Python, a la hora de comparar algo con `None` es recomendable usar `is` o `is not` en vez de `==` o `!=`.  La razón es que si usas los *operadores de igualdad y desigualdad* (`==`, `!=`), existe una posibilidad remota de que no ocurra lo que te estás esperando.

El motivo que podría causar ese comportamiento extraño, lo veremos mucho más adelante.

Esto sólo se aplica a `None` y tampoco es que sea algo crítico.